# 02. 브라질 이커머스(Olist) 데이터 분석 ② — 시간 패턴 · 카테고리 · 배송과 리뷰

01 노트북에서 데이터 구조 · 고객 · 매출을 봤습니다. 이번에는 **언제 · 무엇이 팔리고, 배송이 고객 만족에 어떤 영향을 주는지** 분석합니다.
마지막에 전체 인사이트를 정리하고, 03 노트북에서 이 결과를 웹 대시보드로 만들어 배포합니다.

<div style="border:1px solid #9ABBE0;border-left:5px solid #1F3A7A;background:#EAF2FD;padding:10px 14px;border-radius:4px">

**이 노트북의 학습 목표**

1. `.dt` 로 날짜에서 연 · 월 · 요일 · 시간을 꺼내 **기간별 패턴**을 분석한다.
2. 기간이 다른 데이터를 비교할 때의 함정(**공정한 비교**)을 이해한다.
3. `merge` + `groupby().agg()` 로 카테고리별 판매량 · 매출을 구하고 **판매량 순위와 매출 순위의 차이**를 읽는다.
4. 배송 기간의 대표값(평균 · 중앙값 · 절사평균)을 고르고, **배송 지연과 리뷰 점수의 관계**를 확인한다.

</div>

In [1]:
# ── 준비: 라이브러리 불러오기 ──────────────────────────────
import pandas as pd                    # 표(DataFrame) 다루기
import plotly.express as px            # 짧은 코드로 그래프 그리기
import plotly.graph_objects as go      # 그래프를 세밀하게 꾸미기
import plotly.io as pio

# 그래프 표시 방식: VS Code · JupyterLab · Colab 어디서나 보이도록 설정
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"  # 흰 배경 테마를 기본으로

# 데이터 폴더 위치 — 이 노트북 폴더(notebooks)의 '한 단계 위(..)'에 있는 data 폴더
#   원본 9개는 용량을 줄이려고 압축(.csv.gz)해서 data/ 에 들어 있습니다.
#   pd.read_csv 는 .csv.gz 를 압축을 풀지 않고 그대로 읽습니다.
PATH = "../data/"

pd.set_option("display.max_columns", 30)       # 열이 많아도 생략 없이 보이게
pd.set_option("display.float_format", "{:,.2f}".format)   # 실수는 천 단위 쉼표 + 소수 둘째 자리
print("pandas", pd.__version__)

pandas 3.0.6


## 0. 기본 데이터 준비 (01 노트북 PART C 와 같은 과정)

노트북은 각자 따로 실행할 수 있어야 하므로, 01 에서 만든 **배송 완료 주문 + 주문 금액** 표(`sales`)를 다시 만듭니다.

In [2]:
orders = pd.read_csv(PATH + "olist_orders_dataset.csv.gz")
payments = pd.read_csv(PATH + "olist_order_payments_dataset.csv.gz")

# 날짜 열 4개를 한꺼번에 날짜형으로 변환
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

delivered = orders[orders["order_status"] == "delivered"].copy()                          # 배송 완료 주문만
order_value = payments.groupby("order_id", as_index=False)["payment_value"].sum()         # 주문별 결제 합계
sales = pd.merge(delivered, order_value, on="order_id")                                   # 한 행 = 주문 1건
sales = sales.rename(columns={"order_purchase_timestamp": "purchase_ts"})                 # 긴 열 이름 줄이기
print(sales.shape)
sales[["order_id", "purchase_ts", "payment_value"]].head()

(96477, 9)


,order_id,purchase_ts,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,28.62


---
# PART D. 시간 패턴 분석 (Q5)

## D-1. 일별 주문 수 — 특별한 날 찾기

In [3]:
# resample("D"): 날짜 인덱스를 하루 단위로 묶기 (Grouper 와 같은 역할, 날짜가 인덱스일 때 사용)
daily = sales.set_index("purchase_ts").resample("D").agg(
    orders=("order_id", "count"), revenue=("payment_value", "sum"))
daily = daily.loc["2017-01":"2018-08"]          # 데이터가 온전한 기간만

fig = px.line(daily, y="orders", labels={"purchase_ts": "날짜", "orders": "주문 수"}, title="일별 주문 수")
peak_day = daily["orders"].idxmax()
fig.add_annotation(x=peak_day, y=daily["orders"].max(), text=f"{peak_day:%Y-%m-%d} 블랙프라이데이",
                   showarrow=True, arrowhead=2)
fig.show()

## D-2. 날짜에서 필요한 부분만 꺼내기 — `.dt`

날짜형 열 뒤에 `.dt.` 를 붙이면 연 · 월 · 요일 · 시간 등을 꺼낼 수 있습니다.

| 코드 | 결과 | 예 (2017-11-24 13:05, 금요일) |
|---|---|---|
| `.dt.year` · `.dt.month` · `.dt.day` | 연 · 월 · 일 | 2017 · 11 · 24 |
| `.dt.quarter` | 분기 (1~4) | 4 |
| `.dt.dayofweek` | 요일 번호 (**월=0 … 일=6**) | 4 |
| `.dt.hour` | 시 (0~23) | 13 |
| `.dt.to_period("M")` | 연-월 | 2017-11 |

In [4]:
sales["year"] = sales["purchase_ts"].dt.year
sales["month"] = sales["purchase_ts"].dt.month
sales["quarter"] = sales["purchase_ts"].dt.quarter
sales["weekday"] = sales["purchase_ts"].dt.dayofweek
sales["hour"] = sales["purchase_ts"].dt.hour
sales[["purchase_ts", "year", "month", "quarter", "weekday", "hour"]].head()

,purchase_ts,year,month,quarter,weekday,hour
0,2017-10-02 10:56:33,2017,10,4,0,10
1,2018-07-24 20:41:37,2018,7,3,1,20
2,2018-08-08 08:38:49,2018,8,3,2,8
3,2017-11-18 19:28:06,2017,11,4,5,19
4,2018-02-13 21:18:39,2018,2,1,1,21


## D-3. 연도별 비교 — 공정하게 비교하기

먼저 단순하게 연도별 매출 합계를 구해 봅니다.

In [5]:
by_year = sales.groupby("year")["payment_value"].sum()
months_with_data = sales.groupby("year")["purchase_ts"].apply(lambda s: s.dt.month.nunique())   # 연도마다 데이터가 있는 달 수
pd.DataFrame({"매출 합계": by_year, "데이터가 있는 달 수": months_with_data})

,매출 합계,데이터가 있는 달 수
year,,
2016,"46,586.33",2
2017,"6,922,900.24",12
2018,"8,452,975.20",8


<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**함정** — 2016년은 2개월, 2018년은 8개월치 데이터밖에 없습니다. 이대로 비교하면 "2016년은 망했고 2017년이 2018년보다 좋다" 같은 **틀린 결론**이 나옵니다.
공정하게 비교하려면 **같은 기간끼리** 비교해야 합니다. 2017년과 2018년의 **1~8월**을 비교해 봅니다.

</div>

In [6]:
jan_aug = sales[sales["month"] <= 8]                                  # 두 해 모두 1~8월만
compare = jan_aug[jan_aug["year"].isin([2017, 2018])].groupby("year").agg(
    revenue=("payment_value", "sum"), orders=("order_id", "count"))
growth = (compare.loc[2018] / compare.loc[2017] - 1) * 100            # 증가율(%)
print(compare)
print(f"\n1~8월 기준 증가율 — 매출 {growth['revenue']:.0f}%, 주문 수 {growth['orders']:.0f}%")

          revenue  orders
year                     
2017 3,473,862.76   21998
2018 8,452,975.20   52783

1~8월 기준 증가율 — 매출 143%, 주문 수 140%


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 6 — 같은 1~8월끼리 비교하면 2018년 매출이 2017년보다 약 2.4배(+143%)** 입니다. 회사가 빠르게 성장하는 중이었습니다.

</div>

## D-4. 분기별 비교 — 같은 함정, 반대 결론

분기(Q1~Q4)도 마찬가지입니다. 전체 기간을 그대로 더한 결과와 **1년치가 온전한 2017년만** 본 결과를 나란히 비교합니다.

In [7]:
all_years = sales.groupby("quarter")["payment_value"].sum()                          # 잘못된 방법: 모든 연도를 섞어서 합계
only_2017 = sales[sales["year"] == 2017].groupby("quarter")["payment_value"].sum()   # 올바른 방법: 2017년만

from plotly.subplots import make_subplots     # 그래프 여러 개를 한 그림에 배치
fig = make_subplots(rows=1, cols=2, subplot_titles=("✗ 모든 연도를 섞은 분기 합계", "✓ 2017년만 본 분기 합계"))
fig.add_trace(go.Bar(x=["1Q", "2Q", "3Q", "4Q"], y=all_years.values, marker_color="#999999"), row=1, col=1)
fig.add_trace(go.Bar(x=["1Q", "2Q", "3Q", "4Q"], y=only_2017.values, marker_color="#03588C"), row=1, col=2)
fig.update_layout(showlegend=False, height=380, title="분기별 매출 — 비교 방법에 따라 결론이 정반대")
fig.show()

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**해석** — 왼쪽(잘못된 방법)에서는 4분기가 가장 낮아 보이지만, 그건 2018년 4분기 데이터가 **아예 없고** 2016년 4분기는 거의 0이기 때문입니다.
2017년만 보면 **4분기가 가장 높습니다**(블랙프라이데이 + 꾸준한 성장). 데이터를 묶기 전에 **"각 그룹에 같은 양의 기간이 들어 있나?"** 를 꼭 확인하세요.

</div>

## D-5. 요일별 패턴

요일 번호(0~6)를 이름으로 바꿉니다. 함수 + `apply` 대신 **딕셔너리 + `map`** 을 쓰면 한 줄로 끝납니다.
그리고 `pd.Categorical` 로 **월→일 순서**를 정해 주면 그래프가 가나다순이 아닌 요일 순서로 그려집니다.

In [8]:
day_names = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
sales["weekday_name"] = sales["weekday"].map(day_names)        # 0 → "월", 1 → "화" …
sales["weekday_name"] = pd.Categorical(sales["weekday_name"], categories=list("월화수목금토일"), ordered=True)

by_weekday = sales.groupby("weekday_name", observed=True)["order_id"].count()
fig = px.bar(x=by_weekday.index, y=by_weekday.values, labels={"x": "요일", "y": "주문 수"},
             title="요일별 주문 수 — 월요일이 가장 많고 주말이 적음")
fig.show()

<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**요일은 합계로 비교해도 될까?** 약 20개월 동안 각 요일은 거의 같은 횟수(약 86~87번)씩 들어 있으므로 합계로 비교해도 괜찮습니다.
반면 분기 · 연도는 들어 있는 기간이 달라서 합계 비교가 위험했습니다.

</div>

## D-6. 시간대별 패턴

In [9]:
by_hour = sales.groupby("hour")["order_id"].count()
fig = px.bar(x=by_hour.index, y=by_hour.values, labels={"x": "시 (0~23)", "y": "주문 수"},
             title="시간대별 주문 수 — 오전 10시 ~ 오후 10시가 활발, 새벽 5시가 가장 적음")
fig.update_xaxes(dtick=1)        # 눈금을 1시간마다 표시
fig.show()

## D-7. 요일 × 시간 히트맵

두 기준(요일, 시간)을 한 번에 보려면 **히트맵**을 그립니다. 히트맵은 행 = 요일, 열 = 시간인 **2차원 표(wide 형식)** 가 필요합니다.
`pivot_table(index=행, columns=열, values=값, aggfunc=계산)` 이 긴 표(long 형식)를 2차원 표로 바꿔 줍니다.

In [10]:
heat = sales.pivot_table(index="weekday_name", columns="hour", values="order_id",
                         aggfunc="count", observed=True)
heat.iloc[:, :8]          # 앞 8개 시간대만 미리보기

hour,0,1,2,3,4,5,6,7
weekday_name,,,,,,,,
월,315,132,63,34,21,22,64,157
화,300,151,79,28,28,24,69,216
수,385,171,81,31,33,24,91,208
목,345,165,74,38,30,28,82,211
금,412,210,66,46,39,35,91,203
토,308,167,65,41,25,23,51,102
일,256,137,68,41,27,26,30,102


In [11]:
fig = px.imshow(heat, labels=dict(x="시", y="요일", color="주문 수"),
                color_continuous_scale="Blues", aspect="auto",
                title="요일 × 시간대 주문 수 (진할수록 주문이 많음)")
fig.update_xaxes(dtick=1)
fig.show()

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 7 — 평일 낮~저녁(10~22시)에 주문이 몰리고, 주말과 새벽에는 적습니다.** 특히 평일 오전 10~11시와 오후 2~4시가 진합니다.
프로모션 알림 · 고객센터 인력 배치를 이 시간대에 맞추면 효과적입니다.
(이 히트맵은 두 변수의 **상관계수**가 아니라 **조합별 주문 수**입니다. 용어를 구분하세요.)

</div>

---
# PART E. 카테고리 분석 (Q6)

## E-1. 상품에 카테고리 이름(영어) 붙이기 — left merge

In [12]:
items = pd.read_csv(PATH + "olist_order_items_dataset.csv.gz")
products = pd.read_csv(PATH + "olist_products_dataset.csv.gz")
category = pd.read_csv(PATH + "product_category_name_translation.csv.gz")

print("카테고리가 빈 상품:", products["product_category_name"].isna().sum(), "개")
missing_trans = set(products["product_category_name"].dropna()) - set(category["product_category_name"])
print("영어 번역이 없는 카테고리:", missing_trans)

카테고리가 빈 상품: 610 개
영어 번역이 없는 카테고리: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


`merge` 의 `how` 옵션에 따라 결과가 달라집니다.

| how | 남는 행 | 이번 경우 |
|---|---|---|
| `"inner"` (기본값) | **양쪽 모두에** 있는 행만 | 카테고리가 비었거나 번역이 없는 상품이 **사라짐** |
| `"left"` | **왼쪽 표의 모든 행** (짝이 없으면 빈 값) | 상품이 하나도 사라지지 않음 ✓ |

In [13]:
inner = pd.merge(products, category, on="product_category_name", how="inner")
left = pd.merge(products, category, on="product_category_name", how="left")
print("상품 수:", len(products), "/ inner:", len(inner), "/ left:", len(left))

# 빈 영어 이름 채우기: 번역이 없으면 포르투갈어 이름, 그것도 없으면 'unknown'
left["category"] = (left["product_category_name_english"]
                    .fillna(left["product_category_name"])
                    .fillna("unknown"))

상품 수: 32951 / inner: 32328 / left: 32951


In [14]:
# 판매 상품(items) + 배송 완료 여부 + 카테고리 — 한 행 = 팔린 상품 1개
cat_sales = (items
             .merge(delivered[["order_id"]], on="order_id")               # 배송 완료 주문의 상품만
             .merge(left[["product_id", "category"]], on="product_id", how="left"))
cat_sales[["order_id", "category", "price", "freight_value"]].head()

,order_id,category,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90,18.14


## E-2. `agg` 로 여러 계산을 한 번에

`groupby(...).agg(새이름=("열", "계산"))` 형식(**named aggregation**)을 쓰면 결과 열 이름까지 한 번에 정할 수 있습니다. 먼저 작은 예제로 연습합니다.

In [15]:
toy = pd.DataFrame({"반": ["A", "A", "B", "B", "B"], "점수": [80, 90, 70, 85, 95], "학생": ["가", "나", "다", "라", "마"]})
toy.groupby("반").agg(학생수=("학생", "count"), 평균=("점수", "mean"), 최고=("점수", "max"))

,학생수,평균,최고
반,,,
A,2,85.00,90
B,3,83.33,95


In [16]:
cat = cat_sales.groupby("category").agg(
    items=("order_id", "size"),          # 팔린 상품 개수 (행 수)
    orders=("order_id", "nunique"),      # 주문 건수 (같은 주문에 여러 개 담겨도 1건)
    revenue=("price", "sum"),            # 매출 (상품 가격 합계, 배송비 제외)
    avg_price=("price", "mean"),         # 평균 가격
    freight=("freight_value", "sum"),    # 배송비 합계
)
cat.sort_values("items", ascending=False).head(10)

,items,orders,revenue,avg_price,freight
category,,,,,
bed_bath_table,10953,9272,"1,023,434.76",93.44,"201,774.50"
health_beauty,9465,8647,"1,233,131.72",130.28,"178,957.81"
sports_leisure,8431,7530,"954,852.55",113.25,"163,404.36"
furniture_decor,8160,6307,"711,927.69",87.25,"168,402.23"
computers_accessories,7644,6530,"888,724.61",116.26,"143,999.16"
housewares,6795,5743,"615,628.69",90.60,"142,763.56"
watches_gifts,5859,5495,"1,166,176.98",199.04,"98,156.14"
telephony,4430,4093,"309,860.23",69.95,"69,342.39"
garden_tools,4268,3448,"470,495.28",110.24,"96,650.40"


<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**'판매 수'와 '주문 수'는 다릅니다.** 예를 들어 bed_bath_table(침구 · 욕실)은 상품 약 11,000개가 팔렸지만 주문은 약 9,300건입니다. 한 주문에 수건 여러 장처럼 여러 개를 담았기 때문입니다.
그래프 제목 · 열 이름에 무엇을 셌는지 정확히 적으세요.

</div>

## E-3. 판매량 TOP vs 매출 TOP

In [17]:
top_items = cat.sort_values("items", ascending=False).head(15)
top_rev = cat.sort_values("revenue", ascending=False).head(15)

fig = make_subplots(rows=1, cols=2, subplot_titles=("판매 수량 TOP 15", "매출 TOP 15 (R$)"), horizontal_spacing=0.25)
# 가로 막대: 위에서부터 1위가 오도록 [::-1] 로 순서를 뒤집음
fig.add_trace(go.Bar(y=top_items.index[::-1], x=top_items["items"][::-1], orientation="h", marker_color="#03588C"), row=1, col=1)
fig.add_trace(go.Bar(y=top_rev.index[::-1], x=top_rev["revenue"][::-1], orientation="h", marker_color="#F24472"), row=1, col=2)
fig.update_layout(showlegend=False, height=520, title="카테고리 순위 — 많이 팔린 것과 돈이 되는 것은 다르다")
fig.show()

In [18]:
# 순위 비교표 — rank(): 큰 값일수록 1위
rank = pd.DataFrame({"판매수량 순위": cat["items"].rank(ascending=False).astype(int),
                     "매출 순위": cat["revenue"].rank(ascending=False).astype(int),
                     "평균 가격": cat["avg_price"].round(1)})
rank.sort_values("매출 순위").head(8)

,판매수량 순위,매출 순위,평균 가격
category,,,
health_beauty,2,1,130.30
watches_gifts,7,2,199.00
bed_bath_table,1,3,93.40
sports_leisure,3,4,113.30
computers_accessories,5,5,116.30
furniture_decor,4,6,87.20
housewares,6,7,90.60
cool_stuff,12,8,164.10


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 8 — 판매 수량 1위는 bed_bath_table(침구 · 욕실), 매출 1위는 health_beauty(건강 · 미용)** 입니다.
watches_gifts(시계 · 선물)는 수량 순위는 낮지만 평균 가격이 높아 **매출 2위**입니다. 무엇을 기준으로 '잘 팔린다'고 할지 먼저 정해야 합니다.

</div>

## E-4. 매출 집중도 — 파레토 분석

"매출의 80%는 상위 20% 상품에서 나온다"는 **파레토 법칙**이 맞는지 누적 비율로 확인합니다.

In [19]:
rev_sorted = cat["revenue"].sort_values(ascending=False)
cum_share = rev_sorted.cumsum() / rev_sorted.sum() * 100          # cumsum(): 누적 합계
n80 = (cum_share < 80).sum() + 1                                   # 80% 에 처음 도달하는 순위
print(f"카테고리 {len(cat)}개 중 상위 10개가 매출의 {cum_share.iloc[9]:.1f}%")
print(f"매출 80% 에 도달하려면 상위 {n80}개 카테고리 필요 ({n80 / len(cat) * 100:.0f}%)")

fig = px.line(x=range(1, len(cum_share) + 1), y=cum_share.values, markers=True,
              labels={"x": "카테고리 순위", "y": "누적 매출 비율 (%)"}, title="카테고리 누적 매출 비율")
fig.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80%")
fig.show()

카테고리 74개 중 상위 10개가 매출의 62.4%
매출 80% 에 도달하려면 상위 18개 카테고리 필요 (24%)


In [20]:
# 매출 대비 배송비 비율 — 무겁고 큰 상품일수록 배송비 부담이 큼
cat["freight_ratio"] = cat["freight"] / cat["revenue"] * 100
print(f"전체 배송비 비율: {cat['freight'].sum() / cat['revenue'].sum() * 100:.1f}%")
cat[cat["items"] >= 500].sort_values("freight_ratio", ascending=False)[["items", "avg_price", "freight_ratio"]].head(5)

전체 배송비 비율: 16.6%


,items,avg_price,freight_ratio
category,,,
electronics,2729,56.81,29.46
office_furniture,1668,160.76,25.01
furniture_decor,8160,87.25,23.65
housewares,6795,90.60,23.19
telephony,4430,69.95,22.38


---
# PART F. 배송과 리뷰 분석 (Q7)

## F-1. 배송 기간 계산 — 날짜끼리 빼기

In [21]:
# 날짜 - 날짜 = Timedelta(기간) → .dt.total_seconds() 로 초 단위 숫자 → 86,400(하루의 초)로 나누면 '일'
sales["delivery_days"] = (sales["order_delivered_customer_date"] - sales["purchase_ts"]).dt.total_seconds() / 86400
# 약속한 배송 예정일까지의 기간
sales["promised_days"] = (sales["order_estimated_delivery_date"] - sales["purchase_ts"]).dt.total_seconds() / 86400

ship = sales.dropna(subset=["delivery_days"]).copy()     # 배송 완료일이 기록된 주문만 (8건 제외)
ship["delivery_days"].describe()

count   96,469.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_days, dtype: float64

## F-2. 대표값 고르기 — 평균 · 중앙값 · 절사평균

| 대표값 | 계산 | 특징 |
|---|---|---|
| 평균 (mean) | 모두 더해 개수로 나눔 | 아주 큰 값(이상치)에 끌려감 |
| 중앙값 (median) | 크기 순으로 줄 세운 가운데 값 | 이상치에 거의 영향 없음 |
| 절사평균 (trimmed mean) | 위 · 아래 끝 몇 %를 잘라 내고 평균 | 평균과 중앙값의 절충 |

In [22]:
d = ship["delivery_days"]
low, high = d.quantile([0.05, 0.95])                 # quantile(): 하위 5%, 상위 5% 경계값
trimmed = d[(d >= low) & (d <= high)].mean()         # 양쪽 5%씩 잘라 낸 평균
print(f"평균 {d.mean():.2f}일 / 중앙값 {d.median():.2f}일 / 절사평균(5~95%) {trimmed:.2f}일 / 최댓값 {d.max():.0f}일")

평균 12.56일 / 중앙값 10.22일 / 절사평균(5~95%) 11.55일 / 최댓값 210일


**상자 그림(box plot)으로 이상치 보기** — 상자의 아래 · 위 끝이 Q1(25%) · Q3(75%) 입니다.
**Q3 + 1.5 × IQR**(IQR = Q3 − Q1)보다 큰 값은 점으로 따로 찍히는 **이상치**입니다.

In [23]:
q1, q3 = d.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"이상치 기준: {upper:.1f}일 초과  → 전체의 {(d > upper).mean() * 100:.1f}%")

fig = px.box(ship, y="delivery_days", points="outliers", labels={"delivery_days": "배송 기간 (일)"},
             title="배송 기간 상자 그림")
fig.show()

이상치 기준: 29.2일 초과  → 전체의 5.1%


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**결론 — 대표값은 중앙값(약 10일)을 씁니다.** 최대 200일이 넘는 극단값 때문에 평균(12.6일)이 부풀려져 있습니다.
이상치를 **지워 버리기**보다 **중앙값처럼 이상치에 강한 대표값**을 쓰는 편이 데이터를 잃지 않습니다(지울 때는 이유를 기록).

</div>

## F-3. 월별 배송 기간 추이

In [24]:
ship["ym"] = ship["purchase_ts"].dt.to_period("M").dt.to_timestamp()      # 월의 첫날 날짜로 (그래프 x축용)
period = (ship["ym"] >= "2017-01-01") & (ship["ym"] <= "2018-08-01")      # 데이터가 온전한 20개월
monthly_ship = ship[period].groupby("ym").agg(median_days=("delivery_days", "median"))

fig = px.bar(monthly_ship, y="median_days", text_auto=".1f",
             labels={"ym": "주문 월", "median_days": "배송 기간 중앙값 (일)"}, title="월별 배송 기간 (중앙값)")
fig.update_xaxes(dtick="M1", tickformat="%Y-%m", tickangle=-45)
fig.show()

<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**원인을 단정하지 마세요.** 2018년 2~3월에 배송이 길어졌다가 4월 이후 빨라졌습니다. 하지만 데이터만으로는 '물류 개선' 때문인지, 주문 지역 · 상품 구성이 바뀌어서인지 알 수 없습니다.
또 마지막 몇 달은 **아직 배송 중인(오래 걸리는) 주문이 빠져 있어** 실제보다 빨라 보일 수 있습니다. 그래프에는 관찰한 사실("배송 기간 단축")만 적고, 원인은 추가 분석으로 확인합니다.

</div>

## F-4. 약속을 지켰을까? — 배송 지연율

고객에게 중요한 것은 '며칠 걸렸나'보다 **'약속한 날짜까지 왔나'** 입니다. 실제 배송일이 예정일보다 늦으면 **지연**으로 봅니다.

In [25]:
ship["late"] = ship["order_delivered_customer_date"] > ship["order_estimated_delivery_date"]   # True = 지연
print(f"약속 기간(중앙값): {ship['promised_days'].median():.1f}일 / 실제 배송(중앙값): {ship['delivery_days'].median():.1f}일")
print(f"지연율: {ship['late'].mean() * 100:.1f}%")

late_by_month = ship[period].groupby("ym")["late"].mean() * 100     # True 의 비율 = 지연율(%)
fig = px.line(late_by_month, markers=True, labels={"ym": "주문 월", "value": "지연율 (%)"},
              title="월별 배송 지연율 — 블랙프라이데이(2017.11)와 2018년 2~3월에 급증")
fig.update_layout(showlegend=False)
fig.update_xaxes(dtick="M1", tickformat="%Y-%m", tickangle=-45)
fig.show()

약속 기간(중앙값): 23.2일 / 실제 배송(중앙값): 10.2일
지연율: 8.1%


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 9 — Olist 는 배송 예정일을 넉넉하게(약 23일) 잡아 대부분 지키지만(지연율 약 8%), 주문이 몰린 달에는 지연율이 14~21%까지 치솟습니다.**

</div>

## F-5. 배송이 늦으면 리뷰 점수가 떨어질까?

In [26]:
reviews = pd.read_csv(PATH + "olist_order_reviews_dataset.csv.gz")
print("리뷰 점수 분포(%):")
print((reviews["review_score"].value_counts(normalize=True).sort_index() * 100).round(1))

# 한 주문에 리뷰가 2개 이상인 경우가 있으므로 주문별 평균 점수로 만든 뒤 합침
score = reviews.groupby("order_id", as_index=False)["review_score"].mean()
ship = pd.merge(ship, score, on="order_id", how="inner")

리뷰 점수 분포(%):
review_score
1   11.90
2    3.20
3    8.30
4   19.20
5   57.40
Name: proportion, dtype: float64


In [27]:
by_late = ship.groupby("late").agg(
    orders=("order_id", "count"),
    avg_score=("review_score", "mean"),
    one_star=("review_score", lambda s: (s <= 1).mean() * 100),   # 1점 비율(%) — lambda: 이름 없는 짧은 함수
)
by_late.index = by_late.index.map({False: "제때 도착", True: "지연"})
by_late

,orders,avg_score,one_star
late,,,
제때 도착,88644,4.28,6.78
지연,7825,2.55,46.63


In [28]:
# 배송 기간을 구간으로 나누기 — pd.cut(값, bins=경계, labels=이름)
bins = [0, 7, 14, 21, 30, 60, 1000]
labels = ["~7일", "8~14일", "15~21일", "22~30일", "31~60일", "60일+"]
ship["delivery_group"] = pd.cut(ship["delivery_days"], bins=bins, labels=labels)

by_group = ship.groupby("delivery_group", observed=True)["review_score"].agg(["mean", "count"])
fig = px.bar(by_group, y="mean", text_auto=".2f", labels={"delivery_group": "배송 기간", "mean": "평균 리뷰 점수"},
             title="배송 기간별 평균 리뷰 점수 — 3주가 넘으면 급격히 하락")
fig.update_yaxes(range=[1, 5])
fig.show()

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 10 (가장 중요) — 지연된 주문의 평균 리뷰는 약 2.6점으로, 제때 온 주문(약 4.3점)보다 1.7점 낮습니다.** 지연 주문의 거의 절반(약 47%)이 1점입니다.
배송이 3주를 넘기면 점수가 급격히 떨어집니다. 01 에서 본 **낮은 재구매율(3%)** 을 끌어올리려면 **배송 약속을 지키는 것**이 출발점입니다.

</div>

## F-6. 지역별 배송 기간

In [29]:
customers = pd.read_csv(PATH + "olist_customers_dataset.csv.gz")
ship = pd.merge(ship, customers[["customer_id", "customer_state"]], on="customer_id")

by_state = ship.groupby("customer_state").agg(
    orders=("order_id", "count"),
    median_days=("delivery_days", "median"),
    late_rate=("late", "mean"),
).sort_values("median_days")
by_state["late_rate"] = by_state["late_rate"] * 100

fig = px.bar(by_state, x=by_state.index, y="median_days", color="median_days", color_continuous_scale="Reds",
             labels={"customer_state": "주(state)", "median_days": "배송 기간 중앙값 (일)"},
             title="주(state)별 배송 기간 — 상파울루(SP)가 가장 빠르고 북부 아마존 지역이 가장 느림")
fig.show()

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 11 — 판매자 대부분이 있는 상파울루(SP) 고객은 약 7일, 북부 지역(AM · RR · AP)은 약 25일**이 걸립니다.
먼 지역 고객에게는 예정일을 더 현실적으로 안내하거나 지역 물류 거점을 두는 것을 검토할 수 있습니다.

</div>

---
# 최종 인사이트 정리 (01 + 02)

| 영역 | 발견 | 제안 |
|---|---|---|
| 고객 | 재구매율 **약 3%**, 고객의 42% 가 SP | 재구매 유도(쿠폰 · 추천), 수도권 외 지역 확대 |
| 결제 | 신용카드 75%, 그중 약 3분의 2가 할부 | 무이자 할부 프로모션 |
| 매출 | 1~8월 기준 1년 새 **+143%** 성장, 성장은 객단가가 아닌 **주문 수** 증가 | 신규 고객 유입 채널 유지 |
| 시간 | 블랙프라이데이 하루 1,100건+, 평일 낮 집중 | 행사 전 재고 · 물류 증원 |
| 카테고리 | 수량 1위 침구 · 욕실, 매출 1위 건강 · 미용, 상위 10개가 매출 62% | 핵심 카테고리 집중 관리 |
| 배송 · 리뷰 | 지연 시 리뷰 **4.3 → 2.6점**, 주문 몰린 달 지연율 급증, 북부 25일 | 성수기 물류 대비, 지역별 예정일 현실화 |

다음 **03 노트북**에서는 이 분석을 누구나 웹에서 볼 수 있는 **대시보드**로 만들고 GitHub 와 Streamlit Cloud 로 배포합니다.

# 연습문제

1. 01 노트북에서 했던 월별 매출을 이번에는 `resample("MS")` 로 구해 보세요.
2. 리뷰 점수 1~5점별 **평균 배송 기간(중앙값)** 을 구하세요.
3. 카테고리별 **지연율**을 구하고, 판매 수량 500개 이상인 카테고리 중 지연율이 가장 높은 3곳을 찾으세요.

In [30]:
# 1번 정답 예시
sales.set_index("purchase_ts").resample("MS")["payment_value"].sum().loc["2017-01":"2018-08"].head()

purchase_ts
2017-01-01   127,545.67
2017-02-01   271,298.65
2017-03-01   414,369.39
2017-04-01   390,952.18
2017-05-01   567,066.73
Freq: MS, Name: payment_value, dtype: float64

In [31]:
# 2번 정답 예시
ship.groupby(ship["review_score"].round())["delivery_days"].median()

review_score
1.00   16.73
2.00   13.30
3.00   12.04
4.00   10.79
5.00    9.23
Name: delivery_days, dtype: float64

In [32]:
# 3번 정답 예시 — 상품(cat_sales)과 지연 여부(ship)를 order_id 로 연결
cat_late = cat_sales.merge(ship[["order_id", "late"]], on="order_id")
res = cat_late.groupby("category").agg(items=("order_id", "size"), late_rate=("late", "mean"))
res[res["items"] >= 500].sort_values("late_rate", ascending=False).head(3)

,items,late_rate
category,,
electronics,2729,0.10
unknown,1537,0.09
health_beauty,9462,0.09
